In [4]:
%pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [nltk]  WARNING: The script nltk is installed in '/Library/Frameworks/Python.framework/Versions/3.14/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [nltk]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /Users/admin/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /Users/admin/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /Users/admin/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/admin/nltk_data...


True

In [25]:
import pandas as pd
import glob, os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)

path = r'/Users/admin/Desktop/중요한것/Study-ML-DL-AI/allData/OpinosisDataset1.0/topics'

all_files = glob.glob(os.path.join(path, "*.data"))
filename_list = []
opinion_text = []

for file_ in all_files:
    df = pd.read_table(
        file_,
        header=None,
        encoding='latin1'
    )

    filename = os.path.basename(file_).split('.')[0]

    filename_list.append(filename)

    # 각 행의 문장만 공백으로 이어 붙임
    opinion_text.append(
        ' '.join(df[0].astype(str).tolist())
    )

document_df = pd.DataFrame({
    'filename': filename_list,
    'opinion_text': opinion_text
})

document_df.head(3)

,filename,opinion_text
0,battery-life_ipod_nano_8gb,"short battery life I moved up from an 8gb . I love this ipod except for the battery life . long battery scratch resistant Battery drains even if I don't use it . I only wonder why the battery seems to drain when I'm not using it, even after sliding the top control button to off when shutting down . great in the car, light, portable, good quality, long battery,scratch resistant . 5G lies a more mature iPod, many steps wiser and more able than its one, year, old The iPod gains many incremental improvements, including a brighter screen and better video battery life, but probably the most appealing aspect is the tantalizing price points of $249 for the 30GB version and $349 for the h..."
1,gas_mileage_toyota_camry_2007,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road . It gets great gas mileage . Being a mother who drives a lot I wanted a safe vehicle with good gas mileage and this car delivered that and more . Great gas mileage and comfortable on long trips . Nice looking car and good gas mileage . Good gas mileage, comfortable seating, lots of leg room . The interior is roomy, the ride is smooth and solid and yet it has excellent gas mileage . The gas mileage is still good, cant give specifics but for a V6, its good . I had my OEM Turanzas wear out at 14K and was lucky to get local Bridgestones to gimme a mileage warranty adjustment, got new tires for $..."
2,room_holiday_inn_london,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave us a candy bar and two bottlets of water whic is a very nice touch . We had a room with two double beds which was surprisingly roomy, considering the small hotel rooms I have in previous trips to London . The room was quiet, clean, the bed and pillows were comfortable, and the service was adequate . We arrived about 11 am, room was ready . Room was good size for Europe , clean throughout . The Concier..."


In [26]:
import nltk
import string

from nltk.stem import WordNetLemmatizer

lemmer = WordNetLemmatizer()

def LemTokens(tokens):
    return [lemmer.lemmatize(token) for token in tokens]

remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)

def LemNormalize(text):
    return LemTokens(
        nltk.word_tokenize(
            text.lower().translate(remove_punct_dict)
        )
    )

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english' , \
                             ngram_range=(1,2), min_df=0.05, max_df=0.85)

feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

In [28]:
from sklearn.cluster import KMeans

km_cluster = KMeans(n_clusters=5, max_iter=1000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
cluster_cluster = km_cluster.cluster_centers_

In [29]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,battery-life_ipod_nano_8gb,"short battery life I moved up from an 8gb . I love this ipod except for the battery life . long battery scratch resistant Battery drains even if I don't use it . I only wonder why the battery seems to drain when I'm not using it, even after sliding the top control button to off when shutting down . great in the car, light, portable, good quality, long battery,scratch resistant . 5G lies a more mature iPod, many steps wiser and more able than its one, year, old The iPod gains many incremental improvements, including a brighter screen and better video battery life, but probably the most appealing aspect is the tantalizing price points of $249 for the 30GB version and $349 for the h...",1
1,gas_mileage_toyota_camry_2007,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road . It gets great gas mileage . Being a mother who drives a lot I wanted a safe vehicle with good gas mileage and this car delivered that and more . Great gas mileage and comfortable on long trips . Nice looking car and good gas mileage . Good gas mileage, comfortable seating, lots of leg room . The interior is roomy, the ride is smooth and solid and yet it has excellent gas mileage . The gas mileage is still good, cant give specifics but for a V6, its good . I had my OEM Turanzas wear out at 14K and was lucky to get local Bridgestones to gimme a mileage warranty adjustment, got new tires for $...",4
2,room_holiday_inn_london,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave us a candy bar and two bottlets of water whic is a very nice touch . We had a room with two double beds which was surprisingly roomy, considering the small hotel rooms I have in previous trips to London . The room was quiet, clean, the bed and pillows were comfortable, and the service was adequate . We arrived about 11 am, room was ready . Room was good size for Europe , clean throughout . The Concier...",3
3,location_holiday_inn_london,"Great location for tube and we crammed in a fair amount of sightseeing in a short time . All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price . Great Price Terrific Location . The location is absolutely wonderful . Our main focus before booking was looking for a low price with good location . Very Good Value , Excellent Location . The location is ideal for travel all over the city via the Underground . Excellent hotel in prime location . Good Location Horrible Service Disgusting Procedures . First and foremost, the good, yes it has good location, a block away from Gloucester Road Tube Station . The conve...",0
4,staff_bestwestern_hotel_sfo,"Staff are friendly and helpful . The staff in the morning seemed to need another cup of coffee as they weren't excited to see us . A perfect hotel a perfect location perfect staff ! The front desk staff is more than eager to help, smile, make you happy . The staff is friendly and we only went to one of the &quot Happy Hours&quot for a glass of wine as we were either at the baseball game or on the two tours we took in , busy time ! The front desk and the concierge staff were very helpful when we had a question . The staff is helpful and knowledgable . The room was clean and comfortable and the staff were friendly and helpful . The staff was very helpful, got two great sugges...",3


In [30]:
document_df[document_df['cluster_label']==0].sort_values(by='filename')

,filename,opinion_text,cluster_label
17,food_holiday_inn_london,"The room was packed to capacity with queues at the food buffets . The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances . There are a couple pubs, a great patisserie and even fast food restaurants within a block . The neighborhood is an upscale residential area full of restaurants in easy walking distance ranging from fast food to up, scale with a wide variety of cuisines , , we never had a bad meal . The food was wonderful, a selection of sandwiches, salads, cous cous and spicy wedge potato's, fruit salad and a cake was a good fill up when needed . Full Engli...",0
32,food_swissotel_chicago,"The food for our event was delicious . The food in the lounge was great and very fresh, , , salads, sandwiches etc . As far as food, walk a few blocks toward Michigan Ave turn left or right and there are plently of less expensive places to eat . The Palm resturant in the hotel had some specials Sunday night, we ate there and the food service,etc were outstanding portions are large and we shared since we are not big eaters . Took the charge of the minibar which we had used to keep my 2 year old sons food . We never ate anything onsite so I can't vouch for the food options immediately avialable . The Lobby bar does not serve food very late at night and we couldn't find any vending m...",0
3,location_holiday_inn_london,"Great location for tube and we crammed in a fair amount of sightseeing in a short time . All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price . Great Price Terrific Location . The location is absolutely wonderful . Our main focus before booking was looking for a low price with good location . Very Good Value , Excellent Location . The location is ideal for travel all over the city via the Underground . Excellent hotel in prime location . Good Location Horrible Service Disgusting Procedures . First and foremost, the good, yes it has good location, a block away from Gloucester Road Tube Station . The conve...",0
41,price_amazon_kindle,"If a case was included, as with the Kindle 1, that would have been reflected in a higher price . lower overall price, with nice leather cover . If you have the first Kindle and are deciding whether to upgrade, do it now while you can still get a good price for your old Kindle on Amazon's Marketplace, craigslist, wherever . Before purchasing, I was obsessed with the reviews and predictions I found online and reading about some of the critiques such as the thick border, the lack of touchscreen, lack of battery SD slot, lack of a back light, awkward difficult keyboard layout, minimally faster page flipping, and the super, high price . However, I think all the positives of the K2 greatly...",0
28,price_holiday_inn_london,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price . Great Price Terrific Location . Our main focus before booking was looking for a low price with good location . All for the bargain price off £ 250 for 2 nights including return rail to North Wales . Explore the area and you will find plenty of pubs where you can eat for a reasonable price . Half expected a noisy room by the elevator for the bargain price we paid, but instead got a 17th floor room with awesome city view . The price we paid for our rooms was reasonable . I got this hotel on price line for a 4 night stay with my boyfriend for our anniver...",0
16,service_bestwestern_hotel_sfo,"Both of us having worked in tourism for over 14 years were very disappointed at the level of service provided by this gentleman . The service was good, very friendly staff and we loved the free wine reception each night . Also, thanks to Ken the concierge for recommending Grand Limo service . The restaur

In [31]:
document_df[document_df['cluster_label']==1].sort_values(by='filename')

,filename,opinion_text,cluster_label
33,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate . but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go . This function is not accurate if you don't leave it in battery mode say, when you stop at the Cracker Barrell for lunch and to play one of those trangle games with the tees . It provides immediate alternatives if the route from the online map program was inaccurate or blocked by an obstacle . I've used other GPS units, as well as GPS built into cars and to this day NOTHING beats the accuracy of a Garmin GPS . It got me from point A to point B with 100% accuracy everytime . It has yet to disappoint, getting me everywhere with 100% accurac...",1
9,battery-life_amazon_kindle,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever ! After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there ! NO USER REPLACEABLE BATTERY, , Unless you buy the extended warranty for $65 . After 1 year you pay $80 plus shipping to send the device to Amazon and have the Kindle REPLACED, not the battery changed out . The fact that Kindle 2 has no SD card capability and the battery is not user, serviceable is not an issue with me . Things like the buttons that made it easy to accidentally turn pag...",1
0,battery-life_ipod_nano_8gb,"short battery life I moved up from an 8gb . I love this ipod except for the battery life . long battery scratch resistant Battery drains even if I don't use it . I only wonder why the battery seems to drain when I'm not using it, even after sliding the top control button to off when shutting down . great in the car, light, portable, good quality, long battery,scratch resistant . 5G lies a more mature iPod, many steps wiser and more able than its one, year, old The iPod gains many incremental improvements, including a brighter screen and better video battery life, but probably the most appealing aspect is the tantalizing price points of $249 for the 30GB version and $349 for the h...",1
11,battery-life_netbook_1005ha,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 . Not to mention that as of now Asus will not sell you a spare 3 or 6, cell Li, Ion battery . It also features a N270 cpu, 6, cell 48Wh Li, ion Battery 8 . 3MP webcam, 6, Cell 63Wh Li, ion Battery with a whopping 10 . Realistic battery numbers are between 8 . of battery life if you're using wifi & doing email word processing YouTube web surfing . The Eee Super Hybrid Engine utility lets users overclock or underclock their Eee PC's to boost performance or provide better battery life depending on their immediate requirements . A few other things I'd like to point out is that you must push the micro, sized right ...",1
26,buttons_amazon_kindle,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it . As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately ! After reading through the introductory guide that loads up automatically at the start and following along it took me almost no time to learn which buttons are where and what each of them do . I started reading with the default size without my glasses and noticed I was squinting a bit, so changed to one size larger with a couple button clicks and it was much easier without feeling like the print was too big and took up too mu...",1
34,directions_garmin_nuvi_255W_gps,"You also get upscale features like spoken directions including street names and programmable POIs . I used to hesitate to go out of my directions but not any more . the directions didn't tell me anything I didn't already know after fiddling with it for 10 minutes . It also does not offer an 

In [32]:
document_df[document_df['cluster_label']==2].sort_values(by='filename')

,filename,opinion_text,cluster_label
45,interior_honda_accord_2008,"I love the new body style and the interior is a simple pleasure except for the center dash . However, there are a couple of things that kill it for me 1 terrible driver seat comfort, kills my back 2 lack luster interior design, my Acadia has much better comfort 3 the VCM drives me crazy because the constant change in cylinder use is perceptible enough to be an annoyance . Love the interior and the power and speed, but not hard to beat after what I had . Love the interior and exterior look, the V6 is sensational, and getting compliments on the steel metallic color as if it's a Lexus or BMW . The seats are decent, the interior design is excellent IMO as well as the exterior design, an...",2
22,interior_toyota_camry_2007,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock . 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windshield . Insanely cheap plastic all over interior . Disappointed in interior and exterior quality . I love the color of the exterior and interior . This car is nearly perfect when compared to other cars in this class regarding interior dimensions, visibility, exterior styling, etc . Several parts in the interior rattle including the sunroof and some parts of the dash, the radio randomly...",2
42,quality_toyota_camry_2007,"I previously owned a Toyota 4Runner which had incredible build quality and reliability . I bought the Camry because of Toyota reliability and quality . I purchased a 2007 Camry because of the looks of the redesigned model and because of the legendary Toyota quality and reliability . As of today, I am a bit disappointed in the build quality of the car . Disappointed in interior and exterior quality . Toyota did a great job with design but forgot about quality ! This car needs quality improvement ! The fit and finish in the cabin is not the level of quality I expected . This car looks great and the build quality is good . I am so disappointed in the quality . I've had 2 Camry's be...",2


In [33]:
document_df[document_df['cluster_label']==3].sort_values(by='filename')

,filename,opinion_text,cluster_label
31,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms . The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep . Large comfortable room, wonderful bathroom . The rooms were nice, very comfy bed and very clean bathroom . Bathroom was spacious too and very clean . The bathroom only had a single sink, but it was very large . The room was a standard but nice motel room like any other, bathroom seemed upgraded if I remember . The room was quite small but perfectly formed with a super bathroom . You could eat off the bathroom floor it was so clean . The bathroom do...",3
49,free_bestwestern_hotel_sfo,"The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room . They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day . The service was good, very friendly staff and we loved the free wine reception each night . has free wireless and help you with transportation needs . The nightly free wine tasting from 5 , 6 pm is a brilliant idea and gets guests together to socialise witheach other . They have a happy hour where you have a couple offree drinks between ...",3
39,location_bestwestern_hotel_sfo,"Good Value good location , ideal choice . Great Location , Nice Rooms , Helpless Concierge The location is good, and the overall decor is nice, but there was nothing that I can really rave about . Overall, it was a good location, but an average hotel with serious noise issues . A perfect hotel a perfect location perfect staff ! The location is perfect to pier 39 and surrounding areas . My husband found this hotel on Trip Advisor and it ranked very high and we weren't disappointed at all as it is in a great location and a very nice hotel . The hotel location was perfect . My husband and I stayed for two nights at the Tuscan Inn and enjoyed it's great location to Fisherman Wharf a...",3
50,parking_bestwestern_hotel_sfo,"Parking was expensive but I think this is common for San Fran . there is a fee for parking but well worth it seeing no where to park if you do have a car . The parking was free, which was great, and the hotel was conveniently located for public transport, and local attractions . As for in, and, out parking, I have seen a lot of San Francisco with no car at all . They have a parking garage, but they make you leave your vehicle for them to park and then if you want to take a drive later, you have to wait for the staff to get it . There is no real parking space, so I had to pull up in front of the hotel in a small space . There was valet parking at a cost of $42 . Rooms are very com...",3
2,room_holiday_inn_london,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave us a candy bar and two bottlets of water whic is a very nice touch . We had a room with two double beds which was surprisingly roomy, considering the small hotel rooms I have in previous trips to London . The room was quiet, clean, the bed and pillows were comfortable, and the service was adequate . We arrived about 11 am, room was ready . Room was good size for Europe , clean throughout . The Concier...",3
46,rooms_bestwestern_hotel_sfo,"Great Location , Nice Rooms , Helpless Concierge The rooms were bright, clean and the windows open . Another odd thing we noticed is a majority of the room doors opened OUT into the hallway, rather than into the rooms . The rooms are on the small side but they're still big enough to be comf

In [34]:
document_df[document_df['cluster_label']==4].sort_values(by='filename')

,filename,opinion_text,cluster_label
18,comfort_honda_accord_2008,"Drivers seat not comfortable, the car itself compared to other models of similar class . It's very comfortable, remarkably large inside and just an overall great vehicle . Front seats are very uncomfortable . I'm 6' tall, and find the driving position pretty comfortable . However, there are a couple of things that kill it for me 1 terrible driver seat comfort, kills my back 2 lack luster interior design, my Acadia has much better comfort 3 the VCM drives me crazy because the constant change in cylinder use is perceptible enough to be an annoyance . The seats are extremely uncomfortable . While the Accord is no Acura it is a close relative in terms of quality and comfort . I previ...",4
43,comfort_toyota_camry_2007,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road . Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# . Great gas mileage and comfortable on long trips . Good gas mileage, comfortable seating, lots of leg room . Lots of comfort for the price . The ride is loud and not comfortable . I drive 2 hours to work each day and it is just not comfortable to me . Getting about 26 mpg mixed city hwy with conservative driving, seating 4 people comfortably . The ride is quiet and comfortable . Styling is bland, the engine isn't strong at all, and the car doesn't deliver good comfort...",4
1,gas_mileage_toyota_camry_2007,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road . It gets great gas mileage . Being a mother who drives a lot I wanted a safe vehicle with good gas mileage and this car delivered that and more . Great gas mileage and comfortable on long trips . Nice looking car and good gas mileage . Good gas mileage, comfortable seating, lots of leg room . The interior is roomy, the ride is smooth and solid and yet it has excellent gas mileage . The gas mileage is still good, cant give specifics but for a V6, its good . I had my OEM Turanzas wear out at 14K and was lucky to get local Bridgestones to gimme a mileage warranty adjustment, got new tires for $...",4
35,mileage_honda_accord_2008,"It's quiet, get good gas mileage and looks clean inside and out . The mileage is great, and I've had to get used to stopping less for gas . Thought gas mileage would be better . There are trade offs that I have no problems with, my mileage after two tanks with mostly city driving is 21 but acceleration is very good, smooth, ride a little firm but enjoy the handling . I chose it for the low emissions, value for the money, reliability and gas mileage . The EPA mileage ratings and what the dealer bragged about mileage wise are a joke . 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway . road noise is horrible, stereo sucks, terrible gas mileage etc . Otherwise th...",4
47,performance_honda_accord_2008,"Very happy with my 08 Accord, performance is quite adequate it has nice looks and is a great long, distance cruiser . 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway . Overall performance is good but comfort level is poor . I'm impressed with the performance as well as efficiency gains . It has room, performance, good MPG for its size and excellent reliability . For the record I test, drove the Lexus350 the BMW 5 series, the infiniti G35 and enjoyed the Honda performance equally for far less money ! Very happy with the car enjoy the ride and performance . The performance of the engine is very smooth . This car had rattles at 500 miles and has horrible perfo...",4
29,seats_honda_accord_2008,"Front seats are very uncomfortable . No memory seats, no trip computer, can only display outside temp with trip odometer . needs power seats on the passenger side . I haven't had any back pain from the seats, maybe these people exceed the seat weight limit ? There is a great deal of road noise in the cabin an

In [35]:
from sklearn.cluster import KMeans

km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

document_df['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
50,parking_bestwestern_hotel_sfo,"Parking was expensive but I think this is common for San Fran . there is a fee for parking but well worth it seeing no where to park if you do have a car . The parking was free, which was great, and the hotel was conveniently located for public transport, and local attractions . As for in, and, out parking, I have seen a lot of San Francisco with no car at all . They have a parking garage, but they make you leave your vehicle for them to park and then if you want to take a drive later, you have to wait for the staff to get it . There is no real parking space, so I had to pull up in front of the hotel in a small space . There was valet parking at a cost of $42 . Rooms are very com...",0
27,service_holiday_inn_london,"not customer, oriented hotelvery low service levelboor reception The room was quiet, clean, the bed and pillows were comfortable, and the service was adequate . There were no service issues to speak of, but it's not an overly, friendly hotel . Good Location Horrible Service Disgusting Procedures . If you're looking for a full, service experience, this is still not the hotel for you . The Holiday Inn Kensington Forum is fine if you like large, impersonal hotels with mediocre service, overcrowded breakfast rooms and don't mind tired decor . We found the front desk service unfriendly and indifferent . Great stay friendly service great location . A prompt reply indicated not and that the r...",0
28,price_holiday_inn_london,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price . Great Price Terrific Location . Our main focus before booking was looking for a low price with good location . All for the bargain price off £ 250 for 2 nights including return rail to North Wales . Explore the area and you will find plenty of pubs where you can eat for a reasonable price . Half expected a noisy room by the elevator for the bargain price we paid, but instead got a 17th floor room with awesome city view . The price we paid for our rooms was reasonable . I got this hotel on price line for a 4 night stay with my boyfriend for our anniver...",0
30,rooms_swissotel_chicago,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city . The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio . Great rooms staff and location but extras overpriced . Hotel was very clean and the rooms were comfy . the standard rooms are small, the lobby bar is very small . My son loved the Chicago river view from our 31st floor , rooms where neat and clean . The rooms were small and much more expensive . I've olny ever stayed in the 'standard' rooms in this property, all of wh...",0
20,staff_swissotel_chicago,"The staff at Swissotel were not particularly nice . Each time I waited at the counter for staff for several minutes and then was waved to the desk upon my turn with no hello or anything, or apology for waiting in line . All staff members are very courteous and go the extra mile in meeting your needs during your stay . The staff was great and put us on the 30th floor with a view of the lake . The staff is friendly and knowledgeable about the town and will go out of your way to be helpful . Our whole stay it felt like the staff went out of their way to make sure we had the best visit possible . Great rooms staff and location but extras overpriced . The hotel staff was very nice, f...",0
31,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms . The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep . Large comfortable room, wonderful bathroom . The rooms were

In [36]:
cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape :', cluster_centers.shape)
print(cluster_centers)

cluster_centers shape : (3, 4412)
[[0.         0.         0.00415359 ... 0.         0.00182721 0.00144299]
 [0.0100869  0.00999368 0.00126035 ... 0.00705607 0.         0.        ]
 [0.         0.         0.00086523 ... 0.         0.         0.        ]]


In [37]:
def get_cluster_details(cluster_model, cluster_data, feature_names, cluster_num, top_n_features=10):
    cluster_details = {}

    centrioid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:, ::-1]

    for cluster_num in range(cluster_num):
        cluster_details[cluster_num] = {}
        cluster_details[cluster_num]['cluster'] = cluster_num

        top_features_index = centrioid_feature_ordered_ind[cluster_num, :top_n_features]
        top_features = [feature_names[ind] for ind in top_features_index]

        top_features_values = cluster_model.cluster_centers_[cluster_num, top_features_index].tolist()

        cluster_details[cluster_num]['top_features'] = top_features
        cluster_details[cluster_num]['top_features_value'] = top_features_values
        filenames = cluster_data[cluster_data['cluster_label'] == cluster_num]['filename']
        filenames = filenames.values.tolist()

        cluster_details[cluster_num]['filenames'] = filenames

    return cluster_details

In [38]:
def print_cluster_details(cluster_details):
    for cluster_num, cluster_detail in cluster_details.items():
        print(f'###### Cluster {cluster_num}')
        print('Top features:', cluster_detail['top_features'])
        print('Reviews 파일명:', cluster_detail['filenames'][:7])
        print('===========================================')

In [40]:
feature_names = tfidf_vect.get_feature_names_out()

cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df, feature_names=feature_names, cluster_num=3, top_n_features=10)

print_cluster_details(cluster_details)

###### Cluster 0
Top features: ['room', 'hotel', 'service', 'staff', 'food', 'location', 'bathroom', 'clean', 'price', 'parking']
Reviews 파일명: ['room_holiday_inn_london', 'location_holiday_inn_london', 'staff_bestwestern_hotel_sfo', 'service_swissotel_hotel_chicago', 'service_bestwestern_hotel_sfo', 'food_holiday_inn_london', 'staff_swissotel_chicago']
###### Cluster 1
Top features: ['screen', 'battery', 'keyboard', 'battery life', 'life', 'kindle', 'direction', 'video', 'voice', 'size']
Reviews 파일명: ['battery-life_ipod_nano_8gb', 'voice_garmin_nuvi_255W_gps', 'speed_garmin_nuvi_255W_gps', 'size_asus_netbook_1005ha', 'screen_garmin_nuvi_255W_gps', 'battery-life_amazon_kindle', 'satellite_garmin_nuvi_255W_gps']
###### Cluster 2
Top features: ['interior', 'seat', 'mileage', 'comfortable', 'gas', 'gas mileage', 'transmission', 'car', 'performance', 'quality']
Reviews 파일명: ['gas_mileage_toyota_camry_2007', 'comfort_honda_accord_2008', 'interior_toyota_camry_2007', 'transmission_toyota_camr